In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)

In [ ]:
DATA_PATH = Path("../data/processed/feature_engineered_data.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")

df.head()

In [ ]:
df.info()

In [ ]:
missing_values = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values.head(20)

In [ ]:
df.describe().T

In [ ]:
consumption_column = [
    col for col in df.columns
    if (
        "energy" in col.lower()
        or "consumption" in col.lower()
        or "kwh" in col.lower()
    )
][0]

print("Consumption Column:", consumption_column)

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    df[consumption_column],
    bins=50,
    kde=True
)

plt.title("Energy Consumption Distribution")
plt.xlabel("Consumption")
plt.ylabel("Frequency")

plt.show()

In [ ]:
if "day" in df.columns:

    daily_usage = (
        df.groupby("day")[consumption_column]
        .mean()
    )

    plt.figure(figsize=(14, 6))

    daily_usage.plot()

    plt.title("Average Daily Energy Consumption")
    plt.xlabel("Day")
    plt.ylabel("Average Consumption")

    plt.show()

In [ ]:
if "hour" in df.columns:

    hourly_usage = (
        df.groupby("hour")[consumption_column]
        .mean()
    )

    plt.figure(figsize=(14, 6))

    sns.lineplot(
        x=hourly_usage.index,
        y=hourly_usage.values
    )

    plt.title("Hourly Energy Consumption Pattern")
    plt.xlabel("Hour")
    plt.ylabel("Average Consumption")

    plt.show()

In [ ]:
if "is_peak_hour" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        x="is_peak_hour",
        y=consumption_column,
        data=df
    )

    plt.title("Peak vs Off-Peak Consumption")
    plt.xlabel("Peak Hour")
    plt.ylabel("Consumption")

    plt.show()

In [ ]:
if "season" in df.columns:

    seasonal_usage = (
        df.groupby("season")[consumption_column]
        .mean()
        .sort_values()
    )

    plt.figure(figsize=(10, 5))

    seasonal_usage.plot(kind="bar")

    plt.title("Seasonal Energy Consumption")
    plt.xlabel("Season")
    plt.ylabel("Average Consumption")

    plt.show()

In [ ]:
if "is_weekend" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        x="is_weekend",
        y=consumption_column,
        data=df
    )

    plt.title("Weekend vs Weekday Consumption")

    plt.show()

In [ ]:
if "consumption_intensity" in df.columns:

    intensity_counts = (
        df["consumption_intensity"]
        .value_counts()
    )

    plt.figure(figsize=(8, 5))

    intensity_counts.plot(kind="bar")

    plt.title("Consumption Intensity Distribution")

    plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(14, 10))

sns.heatmap(
    numeric_df.corr(),
    cmap="coolwarm",
    annot=False
)

plt.title("Feature Correlation Heatmap")

plt.show()

In [ ]:
print("KEY INSIGHTS")
print("=" * 50)

peak_hour_avg = (
    df[df["is_peak_hour"] == 1][consumption_column]
    .mean()
)

off_peak_avg = (
    df[df["is_peak_hour"] == 0][consumption_column]
    .mean()
)

print(f"Peak Hour Avg Consumption: {peak_hour_avg:.2f}")
print(f"Off Peak Avg Consumption: {off_peak_avg:.2f}")

seasonal_avg = (
    df.groupby("season")[consumption_column]
    .mean()
    .sort_values(ascending=False)
)

print("\nSeasonal Consumption Ranking:")
print(seasonal_avg)

high_usage_pct = (
    (
        df["consumption_intensity"] == "very_high"
    ).mean() * 100
)

print(f"\nHigh Usage Household Percentage: {high_usage_pct:.2f}%")